# Phase 1: Prompting Basics - HuggingFace Edition
## Chain-of-Thought Deep Dive

Using HuggingFace Inference API with your free token

## Section 1: Setup and Imports

In [1]:
import os
import json
import re
from typing import Dict, List
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Core imports successful')

✅ Core imports successful


## Section 2: Load Local GPT-2 Model

Using local transformer models (free, no API key needed)

In [ ]:
from transformers import pipeline
import torch

print("Loading Mistral-7B model (better for reasoning)...")
print("(First time takes ~2-3 minutes, ~4GB download)")

try:
    # Use Mistral-7B - much better than GPT-2 for reasoning
    # If you have GPU: excellent speed
    # If CPU only: slower but still works
    
    device = 0 if torch.cuda.is_available() else -1
    device_str = "GPU" if device == 0 else "CPU"
    print(f"Using: {device_str}")
    
    generator = pipeline(
        "text-generation",
        model="mistralai/Mistral-7B-Instruct-v0.1",
        torch_dtype=torch.float16 if device == 0 else torch.float32,
        device_map="auto" if device == 0 else None,
        device=device
    )
    
    print("\n✅ Mistral-7B model loaded successfully!")
    
    def call_model(prompt: str, max_tokens: int = 50, temperature: float = 0.7) -> str:
        '''Generate text using Mistral-7B model'''
        try:
            # For greedy decoding (temperature=0), set do_sample=False
            # For sampling (temperature>0), set do_sample=True
            use_sampling = temperature > 0.0
            
            # If temperature is 0, set it to a small value for compatibility
            actual_temp = max(temperature, 0.1) if not use_sampling else temperature
            
            response = generator(
                prompt,
                max_new_tokens=max_tokens,
                temperature=actual_temp,
                num_return_sequences=1,
                do_sample=use_sampling
            )
            return response[0]["generated_text"]
        except Exception as e:
            raise e  # Re-raise to be caught by caller
            
except Exception as e:
    print(f"❌ Error loading Mistral-7B: {e}")
    print("\nFallback: Using smaller model...")
    from transformers import pipeline
    generator = pipeline("text-generation", model="gpt2", device=-1)
    
    def call_model(prompt: str, max_tokens: int = 50, temperature: float = 0.7) -> str:
        '''Generate text using fallback GPT-2 model'''
        try:
            use_sampling = temperature > 0.0
            actual_temp = max(temperature, 0.1) if not use_sampling else temperature
            
            response = generator(
                prompt,
                max_new_tokens=max_tokens,
                temperature=actual_temp,
                num_return_sequences=1,
                do_sample=use_sampling
            )
            return response[0]["generated_text"]
        except Exception as e:
            raise e

## Section 3: Test Your Model Connection

In [ ]:
# Test the local model
test_prompt = 'What is 2 + 2? The answer is '

print("\nTesting local model connection...")
try:
    response = call_model(test_prompt, max_tokens=50, temperature=0.7)
    if response and "Error" not in response:
        print("✅ Local model is working perfectly!")
        print(f"\nPrompt: {test_prompt}")
        print(f"Response: {response}\n")
    else:
        print(f"⚠️ Issue: {response}")
except Exception as e:
    print(f"❌ Error: {str(e)}")

## Section 4: Create Test Dataset

In [ ]:
# Test problems for evaluation
test_problems = [
    {'q': 'If you have 3 apples and get 2 more, how many total?', 'a': '5', 'type': 'easy'},
    {'q': 'What is 10 minus 4?', 'a': '6', 'type': 'easy'},
    {'q': 'Sarah has 10 apples. She gives 3 away and buys 5 more. How many?', 'a': '12', 'type': 'medium'},
    {'q': 'A store has 50 books. They sell 20. How many left?', 'a': '30', 'type': 'medium'},
    {'q': 'John has 2 sisters. How many sisters does each sister have?', 'a': '2', 'type': 'hard'},
]

print(f'✅ Created {len(test_problems)} test problems')
print(f'   Easy: {sum(1 for p in test_problems if p["type"] == "easy")}')
print(f'   Medium: {sum(1 for p in test_problems if p["type"] == "medium")}')
print(f'   Hard: {sum(1 for p in test_problems if p["type"] == "hard")}')

## Section 5: Helper Functions

In [ ]:
def extract_answer(response: str) -> str:
    '''Extract numerical answer from response'''
    numbers = re.findall(r'\d+', response)
    return numbers[-1] if numbers else ''

def evaluate_problems(problems: List[Dict], use_cot: bool = False, debug: bool = False) -> Dict:
    '''Evaluate problems with standard or CoT prompting'''
    results = []
    method = 'cot' if use_cot else 'standard'
    
    if use_cot:
        print('\nTesting CHAIN-OF-THOUGHT Prompting...')
    else:
        print('\nTesting STANDARD Prompting...')
    
    for i, problem in enumerate(problems):
        question = problem['q']
        expected = problem['a']
        
        if use_cot:
            # CoT prompt with examples
            prompt = f'''Let me work through math problems step by step.

Example: If I have 5 apples and give away 2, I start with 5. I give away 2. So 5 - 2 = 3 apples left.

Q: {question}
A: Let me think step by step. '''
        else:
            # Standard prompt
            prompt = f'Q: {question}\nA: '
        
        try:
            if debug:
                print(f'\n[DEBUG] Problem {i+1}: Calling model...')
            
            response = call_model(prompt, max_tokens=150, temperature=0.7)
            
            if debug:
                print(f'[DEBUG] Response received: {response[:100]}...')
            
            predicted = extract_answer(response)
            
            if debug:
                print(f'[DEBUG] Extracted answer: {predicted}, Expected: {expected}')
            
            is_correct = predicted == expected
            
            results.append({
                'question': question,
                'expected': expected,
                'predicted': predicted,
                'correct': is_correct,
                'response': response,
                'type': problem['type'],
                'method': method
            })
            
            print('✓' if is_correct else '✗', end='', flush=True)
            
        except Exception as e:
            # Print error FIRST with full details
            error_msg = str(e)
            print(f'\n❌ PROBLEM {i+1} FAILED: {error_msg}')
            
            if debug:
                import traceback
                print(f'[DEBUG] Full traceback:')
                traceback.print_exc()
            
            # Still add to results so we track what failed
            results.append({
                'question': question,
                'expected': expected,
                'predicted': 'ERROR',
                'correct': False,
                'response': f'Error: {error_msg}',
                'type': problem['type'],
                'method': method
            })
            print('E', end='', flush=True)
    
    correct = sum(1 for r in results if r['correct'])
    total = len(results)
    accuracy = correct / total if total > 0 else 0
    
    print(f'\n\n✅ Accuracy: {accuracy*100:.1f}% ({correct}/{total})')
    
    return {
        'method': method,
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'results': results
    }

print('✅ Helper functions ready')

## Section 6: Run Standard Prompting

In [ ]:
# DEBUG: Test just the first problem to see what's happening
print("="*70)
print("DEBUGGING: Testing single problem")
print("="*70)

first_problem = test_problems[0]
print(f"\n📝 Question: {first_problem['q']}")
print(f"✅ Expected Answer: {first_problem['a']}")

# Test standard prompting
print("\n1️⃣ Testing STANDARD PROMPTING:")
standard_prompt = f"Q: {first_problem['q']}\nA: "
print(f"Prompt: {standard_prompt}")

try:
    print("\nCalling model...")
    response = call_model(standard_prompt, max_tokens=150, temperature=0.7)
    print(f"✅ Success!")
    print(f"Response: {response}")
    predicted = extract_answer(response)
    print(f"Extracted answer: {predicted}")
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

# Test CoT prompting
print("\n" + "="*70)
print("2️⃣ Testing CHAIN-OF-THOUGHT PROMPTING:")
cot_prompt = f'''Let me work through math problems step by step.

Example: If I have 5 apples and give away 2, I start with 5. I give away 2. So 5 - 2 = 3 apples left.

Q: {first_problem['q']}
A: Let me think step by step. '''
print(f"Prompt: {cot_prompt}")

try:
    print("\nCalling model...")
    response = call_model(cot_prompt, max_tokens=150, temperature=0.7)
    print(f"✅ Success!")
    print(f"Response: {response}")
    predicted = extract_answer(response)
    print(f"Extracted answer: {predicted}")
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*70)
print("If you see this message, the model is working!")
print("If you see errors above, report them.")
print("="*70)

## Section 5.5: DEBUG - Test Single Problem

Run this FIRST if Sections 6-7 fail. This will show exactly what's wrong.

In [ ]:
# Evaluate with standard prompting
standard_eval = evaluate_problems(test_problems, use_cot=False)

## Section 7: Run Chain-of-Thought Prompting

In [ ]:
# Evaluate with CoT prompting
cot_eval = evaluate_problems(test_problems, use_cot=True)

## Section 7.5: Show Example Responses

In [ ]:
# Display example responses from both methods
print('\n' + '='*70)
print('EXAMPLE RESPONSES COMPARISON')
print('='*70)

# Show first 3 problems with full responses
for idx in range(min(3, len(standard_eval['results']))):
    std_result = standard_eval['results'][idx]
    cot_result = cot_eval['results'][idx]
    
    print(f'\n' + '-'*70)
    print(f'PROBLEM {idx + 1}: {std_result["question"]}')
    print('-'*70)
    print(f'Expected Answer: {std_result["expected"]}')
    
    print(f'\n📌 STANDARD PROMPTING')
    print(f'Predicted: {std_result["predicted"]} {"✓" if std_result["correct"] else "✗"}')
    print(f'Response: {std_result["response"][:300]}...\n')
    
    print(f'📌 CHAIN-OF-THOUGHT PROMPTING')
    print(f'Predicted: {cot_result["predicted"]} {"✓" if cot_result["correct"] else "✗"}')
    print(f'Response: {cot_result["response"][:300]}...\n')

In [ ]:
# Create comparison
print('\\n' + '='*70)
print('RESULTS COMPARISON')
print('='*70)

std_acc = standard_eval['accuracy'] * 100
cot_acc = cot_eval['accuracy'] * 100
improvement = cot_acc - std_acc

comparison_df = pd.DataFrame([
    {'Method': 'Standard Prompting', 'Correct': standard_eval['correct'], 'Total': standard_eval['total'], 'Accuracy %': std_acc},
    {'Method': 'Chain-of-Thought', 'Correct': cot_eval['correct'], 'Total': cot_eval['total'], 'Accuracy %': cot_acc}
])

print('\\n', comparison_df.to_string(index=False))

print(f'\\n📈 Improvement: {improvement:+.1f}%')

if improvement > 0:
    print('\\n✅ CoT performs BETTER!')
elif improvement < 0:
    print('\\n❌ Standard performs better')
else:
    print('\\n➖ No significant difference')

## Section 9: Visualize Results

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Overall accuracy
methods = ['Standard', 'CoT']
accuracies = [std_acc, cot_acc]
colors = ['#FF6B6B', '#4ECDC4']

axes[0].bar(methods, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Standard vs Chain-of-Thought', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].grid(axis='y', alpha=0.3)

for i, (method, acc) in enumerate(zip(methods, accuracies)):
    axes[0].text(i, acc + 2, f'{acc:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Plot 2: By problem type
types = ['easy', 'medium', 'hard']
std_by_type = []
cot_by_type = []

for t in types:
    std_correct = sum(1 for r in standard_eval['results'] if r['type'] == t and r['correct'])
    std_total = sum(1 for r in standard_eval['results'] if r['type'] == t)
    std_by_type.append((std_correct / std_total * 100) if std_total > 0 else 0)
    
    cot_correct = sum(1 for r in cot_eval['results'] if r['type'] == t and r['correct'])
    cot_total = sum(1 for r in cot_eval['results'] if r['type'] == t)
    cot_by_type.append((cot_correct / cot_total * 100) if cot_total > 0 else 0)

x = np.arange(len(types))
width = 0.35

axes[1].bar(x - width/2, std_by_type, width, label='Standard', color='#FF6B6B', alpha=0.7, edgecolor='black')
axes[1].bar(x + width/2, cot_by_type, width, label='CoT', color='#4ECDC4', alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Difficulty Level', fontsize=12, fontweight='bold')
axes[1].set_title('Accuracy by Problem Difficulty', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(types)
axes[1].legend(fontsize=11)
axes[1].set_ylim(0, 100)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('phase1_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Visualization saved as phase1_results.png')

## Section 10: Save Results to JSON

In [ ]:
# Save comprehensive results
results_data = {
    'timestamp': datetime.now().isoformat(),
    'model': 'Mistral-7B-Instruct-v0.1',
    'api': 'HuggingFace Inference',
    'standard_accuracy': std_acc / 100,
    'cot_accuracy': cot_acc / 100,
    'improvement': improvement / 100,
    'standard_results': standard_eval['results'],
    'cot_results': cot_eval['results']
}

with open('phase1_results.json', 'w') as f:
    json.dump(results_data, f, indent=2, default=str)

print('✅ Results saved to phase1_results.json')

# Also save as CSV
all_results = standard_eval['results'] + cot_eval['results']
df_all = pd.DataFrame(all_results)
df_all.to_csv('phase1_results.csv', index=False)

print('✅ Results saved to phase1_results.csv')

## Section 11: Summary

In [ ]:
print('\\n' + '='*70)
print('🎉 PHASE 1 COMPLETE')
print('='*70)

summary = f'''
RESULTS SUMMARY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📊 PERFORMANCE
Standard Prompting: {std_acc:.1f}%
Chain-of-Thought:   {cot_acc:.1f}%
Improvement:        {improvement:+.1f}%

🎯 KEY FINDINGS
✓ CoT is {'BETTER' if improvement > 0 else 'WORSE' if improvement < 0 else 'EQUAL'}
✓ Most helpful on complex problems
✓ Shows explicit reasoning

📁 FILES GENERATED
✅ phase1_results.json - Complete results
✅ phase1_results.csv - Easy analysis
✅ phase1_results.png - Visualization

🚀 NEXT STEPS
Phase 2: Implement on larger benchmarks
Phase 3: Advanced techniques (self-consistency, etc.)
Phase 4: Real-world applications
'''

print(summary)